# 03 — Turning Crank Words (Free vs Toward a Target)

A **turn** is one small update to a crank word — like taking one optimization step in compressed space.

We compare two styles:

1. **Unconstrained turn** (`turn`) — nudge the word using Crankl’s built-in dynamics (favoring stable/“energy-preserving” motion).
2. **Target-guided turn** (`turn_toward`) — nudge while trying to make the rebuilt 8×8 grid closer to a target grid.

You will see:

- the original packed words
- how total reconstruction error changes over a few steps
- how many slots changed and by how much (Hamming distance)

Because values inside a crank word are coarse (only certain discrete choices), loss may not drop on *every* single step. That is expected.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import CranklAPI, prepare_demo_files

api = CranklAPI()
demo = prepare_demo_files()
original_words = api.pack(demo.source_blocks)

# Keep independent copies so both strategies begin from exactly the same words.
unconstrained_words = original_words.copy()
guided_words = original_words.copy()
learning_rate = 0.04
steps = 5

print("Original words:", [f"0x{word:016x}" for word in original_words])
print("Initial per-block target losses:", [
    api.reconstruction_loss(int(word), target)
    for word, target in zip(original_words, demo.target_blocks)
])

## Run both update styles for a few steps

Each loop calls the real C API once per slot per step. Guided updates also get the matching 64-float target block.

We track **total target reconstruction loss** after every step so you can watch the trajectory, not only the final words.

In [ ]:
def total_target_loss(words):
    return sum(
        api.reconstruction_loss(int(word), target)
        for word, target in zip(words, demo.target_blocks)
    )

unconstrained_history = [total_target_loss(unconstrained_words)]
guided_history = [total_target_loss(guided_words)]

for _ in range(steps):
    unconstrained_words = np.array(
        [api.turn(int(word), learning_rate) for word in unconstrained_words],
        dtype=np.uint64,
    )
    guided_words = np.array(
        [
            api.turn_toward(int(word), target, learning_rate)
            for word, target in zip(guided_words, demo.target_blocks)
        ],
        dtype=np.uint64,
    )
    unconstrained_history.append(total_target_loss(unconstrained_words))
    guided_history.append(total_target_loss(guided_words))

print("Step | unconstrained loss | guided loss")
for step, (free_loss, target_loss) in enumerate(zip(unconstrained_history, guided_history)):
    print(f"{step:>4} | {free_loss:>18.6f} | {target_loss:>11.6f}")

## See what changed

The **diff** helpers answer:

- how many slots differ between two word lists
- the normalized bit-flip distance (Hamming) across the 64-bit words

**Metrics** summarize the population of values inside the resulting archive (density, entropy, energy, etc.).

In [ ]:
free_changed, free_hamming = api.diff(original_words, unconstrained_words)
guided_changed, guided_hamming = api.diff(original_words, guided_words)

print("Unconstrained final words:", [f"0x{word:016x}" for word in unconstrained_words])
print("Guided final words:       ", [f"0x{word:016x}" for word in guided_words])
print(f"\nUnconstrained diff: {free_changed} slots, hamming={free_hamming:.6f}")
print(f"Guided diff:        {guided_changed} slots, hamming={guided_hamming:.6f}")
print("\nGuided archive metrics:")
print(json.dumps(api.metrics(guided_words), indent=2))

## Full file-based finetune

`crankl finetune` is the CLI version of “keep turning toward a target.” It can also use optional calibration vectors, saves a snapshot after each step, and lets you compare the baseline archive to the tuned one.

Same source / target / calibration floats as the other notebooks.

In [ ]:
from crankl_demo import ARTIFACT_DIR, run_cli

baseline_archive = ARTIFACT_DIR / "finetune_baseline.crank"
finetuned_archive = ARTIFACT_DIR / "finetune_result.crank"

run_cli("pack", "--input", demo.source_path, "-o", baseline_archive)
finetune_report = run_cli(
    "finetune", "--input", baseline_archive,
    "--target", demo.target_path,
    "--calib-x", demo.calibration_x_path,
    "--calib-y", demo.calibration_y_path,
    "--steps", 6, "--lr", 0.03,
    "-o", finetuned_archive, "--json",
    expect_json=True,
)
comparison_report = run_cli(
    "compare", baseline_archive, finetuned_archive, "--json", expect_json=True
)

print("\nFinetune losses:")
print(json.dumps(finetune_report, indent=2))
print("\nBaseline-to-finetuned comparison:")
print(json.dumps(comparison_report, indent=2))